# IIIF: what the manifest and the image API make possible

IIIF (the International Image Interoperability Framework, pronounced "triple-eye-eff") is a set of shared specifications for describing and serving images and other digitised objects over the web. Many libraries, archives and museums publish their collections this way, including the institutions that feed Europeana.

This notebook looks at what the standard allows, in four parts:

1. **The manifest**: how a multi-page object such as a book is described as an ordered list of canvases, and how images are attached to those canvases.
2. **The image API**: how a server returns an image at the size, region and quality you ask for, and how that makes deep zoom possible.
3. **Annotations**: how a note can point at a region of a page without touching the image.
4. **A case study**: how Fragmentarium describes a Book of Hours whose leaves are held by many different owners.

The code needs an internet connection, because it talks to live IIIF servers. Everything is read-only.

## 1. Two APIs and one structure

IIIF consists of several specifications. Two matter for this notebook:

| API | Question it answers | What you get |
|---|---|---|
| **Presentation API** | What is this object, and how is it organised? | A JSON document called a *manifest* |
| **Image API** | Give me this image, in this form. | Pixels, selected by a URL |

The Presentation API describes structure. The Image API delivers pixels. The manifest points at image services, so a viewer can start from a single manifest URL and find everything else.

### The structure of a manifest

![IIIF structure: collection, manifest, ranges, canvases, images](../misc/assets/img/iiif_structure.svg)

Reading the diagram from the outside in:

- <b style="color:darkgray">Collection</b> is a list of manifests (or of other collections).
- <b style="color:rosybrown">Manifest</b> describes one object, for example a book. It carries the title, rights statement, attribution and other metadata, and it is the unit you share: one URL stands for the whole object.
- <b style="color:crimson">Canvas</b> is a virtual page: an empty rectangle with a width, a height and a label such as "fol. 1r". Canvases are listed in reading order, and that list order *is* the page order.
- <b style="color:blueviolet">Image</b> is not part of the canvas itself. It is attached to the canvas by an annotation with the motivation <code style="color:blueviolet; border:1.5pt solid blueviolet; padding:1px 2px; border-radius:5px;background:#FFFFFF">painting</code>, which says "place this image onto this canvas". The image can be hosted anywhere, so neighbouring pages of one book may come from different servers, as in the diagram (Yale, Smith College, BnF).
- <b style="color:coral">Range</b> is an optional grouping of canvases, such as a chapter or a section. Ranges give a table of contents without changing the canvas order.

The separation between canvas and image is what lets a manifest organise pages without owning any pixels.

## Setup

In [ ]:
import html
import json
import math
import re
import time
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
from io import BytesIO

import warnings
import requests
from IPython.display import HTML, Image as IPImage, display
from PIL import Image, ImageDraw

warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")
session = requests.Session()
session.headers["User-Agent"] = "dh-workshop-iiif-notebook"


def get_json(url):
    r = session.get(url, timeout=30)
    r.raise_for_status()
    return r.json()


def fetch(url):
    # Download a URL and return the raw bytes.
    r = session.get(url, timeout=60)
    r.raise_for_status()
    return r.content


def pil(url):
    # Download an image URL and return a PIL image together with its transfer size in bytes.
    data = fetch(url)
    return Image.open(BytesIO(data)), len(data)


def kb(n):
    return f"{n / 1024:,.0f} KB" if n < 1024 * 1024 else f"{n / 1024 / 1024:.1f} MB"

## 2. Reading a manifest

The example is a small book from the IIIF Cookbook: five pages from a French printed book digitised by the Bibliothèque nationale de France (Gallica). The manifest is a few kilobytes of JSON. The page images are served from a separate IIIF image server.

In [ ]:
BOOK_URL = "https://iiif.io/api/cookbook/recipe/0009-book-1/manifest.json"
book = get_json(BOOK_URL)

def first(label_map):
    # IIIF 3 labels are language maps: {"en": ["Title"]}. Return the first value.
    return next(iter(label_map.values()))[0]

print("type:  ", book["type"])
print("label: ", first(book["label"]))
print("hint:  ", book.get("behavior"), " (behavior 'paged' tells viewers to show it as a book)")
print("pages: ", len(book["items"]))

`items` holds the canvases in reading order. For every canvas we can follow the chain *canvas → annotation page → painting annotation → image body → image service*:

In [ ]:
def canvas_image(canvas):
    # Follow canvas -> AnnotationPage -> Annotation -> body, and return the body and its image service.
    anno = canvas["items"][0]["items"][0]
    assert anno["motivation"] == "painting"
    body = anno["body"]
    service = body["service"][0] if isinstance(body["service"], list) else body["service"]
    return body, service["id"]


rows = ""
for i, cv in enumerate(book["items"], 1):
    body, svc = canvas_image(cv)
    rows += (f"<tr><td>{i}</td><td>{html.escape(first(cv['label']))}</td>"
             f"<td>{cv['width']} × {cv['height']}</td><td><code>…{svc[-24:]}</code></td></tr>")
display(HTML(
    "<table><tr><th>#</th><th>canvas label</th><th>canvas size</th><th>image service (end of URL)</th></tr>"
    + rows + "</table>"))

One canvas, trimmed to the essentials, shows the same chain as JSON:

In [ ]:
cv = book["items"][2]
body, svc = canvas_image(cv)
trimmed = {
    "id": cv["id"], "type": "Canvas", "label": cv["label"],
    "width": cv["width"], "height": cv["height"],
    "items": [{"type": "AnnotationPage", "items": [{
        "type": "Annotation", "motivation": "painting",
        "body": {"type": "Image", "service": [{"id": svc, "type": "ImageService3"}]},
        "target": cv["id"],
    }]}],
}
print(json.dumps(trimmed, indent=2))

Three things to notice:

- `width` and `height` on the canvas define a coordinate space. They are abstract units, and they are the reference for everything placed on the canvas, including annotations (section 4).
- The painting annotation's `target` is the canvas, and its `body` is the image. That is the whole link between the two.
- The only thing the manifest knows about the pixels is the address of an image service. The next section shows what that address can do.

A viewer works through the same steps. It reads the manifest, lays out the canvases in order, and for each visible canvas asks the image service for the pixels it needs. The thumbnail strip below does exactly that, one small request per page:

In [ ]:
def thumb_strip(canvases, height=220):
    tiles = ""
    for cv in canvases:
        _, svc = canvas_image(cv)
        tiles += (f"<figure style='margin:0;text-align:center;font:12px sans-serif'>"
                  f"<img src='{svc}/full/,{height}/0/default.jpg' style='height:{height}px;border:1px solid #ccc'>"
                  f"<figcaption>{html.escape(first(cv['label']))}</figcaption></figure>")
    display(HTML(f"<div style='display:flex;gap:12px;flex-wrap:wrap'>{tiles}</div>"))

thumb_strip(book["items"])

## 3. The image API: any size, any region

An IIIF image service answers URLs of a fixed form:

```
{service}/{region}/{size}/{rotation}/{quality}.{format}
```

![Anatomy of an IIIF image URL](../misc/assets/img/iiif_image_url.svg)

In the example above, the server is asked for the rectangle starting at pixel (1000, 500) that is 800 px wide and 600 px high, scaled to 800 px wide, rotated 90° clockwise, in colour, as a JPEG.

| Segment | Examples | Meaning |
|---|---|---|
| <code style="color:rosybrown">region</code> | `full`, `square`, `x,y,w,h`, `pct:x,y,w,h` | which part of the image (pixels, or percentages) |
| <code style="color:coral">size</code> | `max`, `400,` (width), `,300` (height), `!400,300` (fit inside), `pct:50` | how large the returned picture is |
| <code style="color:crimson">rotation</code> | `0`, `90`, `!0` | clockwise degrees; `!` mirrors first |
| <code style="color:blueviolet">quality</code> | `default`, `gray`, `bitonal` | colour treatment |
| <code style="color:blueviolet">format</code> | `jpg`, `png`, … | file format |

The server does the cropping, scaling and re-encoding, so the client never has to download the original file in order to look at part of it. A companion document, `info.json`, sits next to every service and states what the server can do:

In [ ]:
PAGE = book["items"][2]                   # the title page
_, SERVICE = canvas_image(PAGE)
info = get_json(f"{SERVICE}/info.json")

print(json.dumps(info, indent=2))

`width` and `height` give the size of the original image in pixels, and `tiles` describes how a viewer may cut it into pieces (used in the zoom section below).

### 3.1 The same image in different sizes

Four requests to the same service, changing only the `size` segment. The table reports the pixels returned and the bytes transferred:

In [ ]:
sizes = [("100 px wide", "100,"), ("400 px wide", "400,"), ("1000 px wide", "1000,"), ("full size", "max")]
results = []
for label, size in sizes:
    url = f"{SERVICE}/full/{size}/0/default.jpg"
    im, n = pil(url)
    results.append((label, url, im, n))

display(HTML("<table><tr><th>request</th><th>pixels returned</th><th>transferred</th></tr>" + "".join(
    f"<tr><td><code>full/{u.split('/full/')[1].split('/')[0]}/0/default.jpg</code></td>"
    f"<td>{im.width} × {im.height}</td><td>{kb(n)}</td></tr>" for _, u, im, n in results) + "</table>"))

display(HTML("<div style='display:flex;gap:12px;align-items:flex-end'>" + "".join(
    f"<figure style='margin:0;font:12px sans-serif;text-align:center'>"
    f"<img src='{u}' style='height:{min(im.height, 300) if im.height < 300 else 300}px'><figcaption>{l}</figcaption></figure>"
    for l, u, im, n in results[:3]) + "</div>"))

The smallest version is several hundred times smaller than the full-size file, and it was produced on request. The institution only needs to keep one high-resolution master.

### 3.2 Regions, rotation and quality

The `region` segment selects a rectangle of the original image in pixel coordinates (`x,y,w,h`, origin at the top left). Combined with the other segments, one image yields many views:

In [ ]:
W, H = info["width"], info["height"]
views = [
    ("full, 260 px wide", "full", "260,", "0", "default"),
    ("region: title line", f"{W//9},{H//5},{W*7//9},{H//6}", "260,", "0", "default"),
    ("region: the vignette", f"{W*28//100},{H*45//100},{W*33//100},{H*23//100}", "260,", "0", "default"),
    ("square (centred crop)", "square", "260,", "0", "default"),
    ("rotated 90°", "full", ",260", "90", "default"),
    ("mirrored", "full", "260,", "!0", "default"),
    ("grey", "full", "260,", "0", "gray"),
]
cells = ""
for title, region, size, rot, q in views:
    url = f"{SERVICE}/{region}/{size}/{rot}/{q}.jpg"
    cells += (f"<figure style='margin:0;font:12px sans-serif;text-align:center;max-width:270px'>"
              f"<img src='{url}' style='max-width:260px;max-height:300px;border:1px solid #ccc'>"
              f"<figcaption><b>{title}</b><br><code style='font-size:10px;word-break:break-all'>/{region}/{size}/{rot}/{q}.jpg</code>"
              f"<br><a href='{url}' target='_blank'>open in browser</a></figcaption></figure>")
display(HTML(f"<div style='display:flex;gap:14px;flex-wrap:wrap;align-items:flex-start'>{cells}</div>"))

print("Full URLs (paste any of them into a browser):\n")
for title, region, size, rot, q in views:
    print(f"{title}:\n  {SERVICE}/{region}/{size}/{rot}/{q}.jpg")

Every view is an ordinary URL, so it can be cited, bookmarked, put into a web page or fetched by a script. Editing a number in the URL is all that is needed to change the view.

**Try it:** change the values below (region, size, rotation, quality) and run the cell again. The cell prints the full URL, which you can copy into a browser address bar to get the same image.

In [ ]:
# Change any of these and run the cell again.
x, y, w, h = 900, 2100, 1050, 1050      # region in pixels of the full-size image
size       = "!500,500"                 # e.g. "500,"  ",500"  "!500,500"  "pct:25"  "max"
rotation   = "0"                        # e.g. "0"  "90"  "180"  "270"  "!0" (mirrored)
quality    = "default"                  # "default"  "gray"  "bitonal"

url = f"{SERVICE}/{x},{y},{w},{h}/{size}/{rotation}/{quality}.jpg"
print("Open this URL in your own browser:")
print(url)
IPImage(url=url)

### 3.3 Deep zoom with tiles

A page like the one above has about 15 million pixels. A viewer on a laptop screen shows perhaps half a million at once, so loading the whole image is wasteful, and for very large scans (maps, newspapers, paintings at gigapixel size) it is not feasible.

Viewers solve this with the same region and size parameters. `info.json` advertises a tile size (`512`) and a list of `scaleFactors` (`1, 2, 4, 8`), which describe zoom levels:

| scale factor | meaning | tile covers (in original pixels) |
|---|---|---|
| 8 | image shown at 1/8 size, most zoomed out | 4096 × 4096 |
| 4 | 1/4 size | 2048 × 2048 |
| 2 | 1/2 size | 1024 × 1024 |
| 1 | full resolution, most zoomed in | 512 × 512 |

At every level a tile is a 512-pixel-wide picture, so the cost of filling the screen stays roughly constant however large the original is. The function below does what a viewer does: it works out which tiles overlap a 512 × 512 pixel window, requests only those, and joins them together.

In [ ]:
def render_viewport(service, info, center, scale, view=512):
    # Fetch just the tiles that cover a view x view pixel window at the given scale factor.
    # center is in full-resolution pixel coordinates. Returns the picture, the number of tiles and the bytes fetched.
    W, H = info["width"], info["height"]
    tile = info["tiles"][0]["width"]
    span = tile * scale                              # original pixels covered by one tile
    rw, rh = view * scale, view * scale              # original pixels covered by the window
    x0 = int(min(max(center[0] - rw / 2, 0), max(W - rw, 0)))
    y0 = int(min(max(center[1] - rh / 2, 0), max(H - rh, 0)))
    x1, y1 = min(x0 + rw, W), min(y0 + rh, H)

    cols = range(x0 // span, (x1 - 1) // span + 1)
    rows = range(y0 // span, (y1 - 1) // span + 1)
    jobs = []
    for r in rows:
        for c in cols:
            sx, sy = c * span, r * span
            sw, sh = min(span, W - sx), min(span, H - sy)
            out_w = math.ceil(sw / scale)
            jobs.append((c, r, f"{service}/{sx},{sy},{sw},{sh}/{out_w},/0/default.jpg"))

    with ThreadPoolExecutor(8) as ex:
        fetched = list(ex.map(lambda j: pil(j[2]), jobs))

    mosaic = Image.new("RGB", (len(cols) * tile, len(rows) * tile), "white")
    total = 0
    for (c, r, _), (im, n) in zip(jobs, fetched):
        mosaic.paste(im.convert("RGB"), ((c - cols[0]) * tile, (r - rows[0]) * tile))
        total += n
    left = (x0 - cols[0] * span) // scale
    top = (y0 - rows[0] * span) // scale
    out = mosaic.crop((left, top, left + math.ceil((x1 - x0) / scale), top + math.ceil((y1 - y0) / scale)))
    return out, len(jobs), total


focus = (1425, 2640)     # a point on the circular vignette, in full-resolution pixels
shots = []
for scale in (8, 4, 2, 1):
    img, n_tiles, n_bytes = render_viewport(SERVICE, info, focus, scale)
    shots.append((scale, img, n_tiles, n_bytes))

strip = Image.new("RGB", (sum(s[1].width for s in shots) + 12 * (len(shots) - 1), max(s[1].height for s in shots)), "white")
x = 0
for _, img, _, _ in shots:
    strip.paste(img, (x, 0)); x += img.width + 12
display(strip)

full_n = len(fetch(f"{SERVICE}/full/max/0/default.jpg"))
display(HTML("<table><tr><th>scale factor</th><th>tiles requested</th><th>transferred</th></tr>" + "".join(
    f"<tr><td>{s}</td><td>{t}</td><td>{kb(b)}</td></tr>" for s, _, t, b in shots)
    + f"</table><p>For comparison, the complete image at full size is {kb(full_n)}.</p>"))

The four pictures are the same 512 × 512 pixel window at increasing zoom, and each was assembled from a handful of tiles. A real viewer such as Mirador repeats this as you pan and zoom, and it can do so for images thousands of times larger, because it only ever requests what is on screen.

Two details worth knowing:

- Servers advertise a **compliance level** in `info.json` (`profile`). Level 0 only offers pre-cut tiles, while levels 1 and 2 accept arbitrary regions and sizes. Level 2 is what makes the free-form requests in section 3.2 possible on any server.
- IIIF image services exist in versions 2 and 3, which differ in small ways (for example `max` replaces `full` as the size keyword). Requests such as `,400` work on both.

## 4. Annotations on a region

Because a canvas has a coordinate space, a note can be attached to a *part* of it. The target of such an annotation is the canvas URL followed by a media fragment, `#xywh=x,y,w,h`:

```
https://iiif.io/api/cookbook/recipe/0009-book-1/canvas/p3#xywh=900,2100,1050,1050
```

The annotation is a separate piece of JSON, following the W3C Web Annotation model. It is not stored in the image, and the image is not modified. Anyone can publish annotations about a canvas they do not own, and a viewer can overlay them when it is given the URL of the annotation page. Below are two annotations on the title page, with the coordinates written in canvas units:

In [ ]:
CANVAS_ID = PAGE["id"]

def make_annotation(canvas_id, region, text, n, motivation="commenting"):
    x, y, w, h = region
    return {
        "id": f"https://example.org/annotations/{n}",
        "type": "Annotation",
        "motivation": motivation,
        "body": {"type": "TextualBody", "language": "en", "format": "text/plain", "value": text},
        "target": f"{canvas_id}#xywh={x},{y},{w},{h}",
    }

REGIONS = [
    ((900, 2100, 1050, 1050), "Circular vignette showing a figure in a tree."),
    ((230, 1520, 510, 500), "Red library stamp."),
]
annotation_page = {
    "@context": "http://iiif.io/api/presentation/3/context.json",
    "id": "https://example.org/annotations/title-page",
    "type": "AnnotationPage",
    "items": [make_annotation(CANVAS_ID, r, t, i + 1) for i, (r, t) in enumerate(REGIONS)],
}
print(json.dumps(annotation_page, indent=2))

The `target` is all that locates the note. To check the coordinates, draw them over a small version of the page. The overlay is drawn locally, on a copy; the server's image stays as it was:

In [ ]:
body, _ = canvas_image(PAGE)
thumb_h = 640
thumb, _ = pil(f"{SERVICE}/full/,{thumb_h}/0/default.jpg")
k = thumb_h / PAGE["height"]                       # canvas units -> thumbnail pixels
draw = ImageDraw.Draw(thumb)
for anno in annotation_page["items"]:
    x, y, w, h = map(int, anno["target"].split("#xywh=")[1].split(","))
    draw.rectangle([x * k, y * k, (x + w) * k, (y + h) * k], outline=(220, 20, 60), width=3)
display(thumb)

Reading the annotation back is the reverse operation: take the region from the target and use it in an Image API request to retrieve exactly the annotated pixels. In this manifest the canvas and the image have the same dimensions. In general they can differ, so the conversion multiplies by the ratio of image size to canvas size:

In [ ]:
def canvas_to_image_region(region, canvas, body):
    kx, ky = body["width"] / canvas["width"], body["height"] / canvas["height"]
    x, y, w, h = region
    return tuple(round(v) for v in (x * kx, y * ky, w * kx, h * ky))

tiles = ""
for anno in annotation_page["items"]:
    region = tuple(int(v) for v in anno["target"].split("#xywh=")[1].split(","))
    ix, iy, iw, ih = canvas_to_image_region(region, PAGE, body)
    url = f"{SERVICE}/{ix},{iy},{iw},{ih}/!300,300/0/default.jpg"
    tiles += (f"<figure style='margin:0;font:12px sans-serif;text-align:center'><img src='{url}' style='border:1px solid #ccc'>"
              f"<figcaption>{html.escape(anno['body']['value'])}<br><code>{ix},{iy},{iw},{ih}</code></figcaption></figure>")
display(HTML(f"<div style='display:flex;gap:16px'>{tiles}</div>"))

This is how region annotations are used in practice: transcriptions of a single line, labels on parts of a map, tags on a detail of a painting, or links from a figure to a catalogue entry. They are stored once, they refer to a canvas by URL, and any IIIF viewer or script can read them.

## 5. Viewing a manifest in Mirador

[Mirador](https://projectmirador.org/) is an open-source IIIF viewer. It is a JavaScript application, so it is not installed with pip. The helper below embeds it from a CDN inside the notebook. It is only given a manifest URL and works out the rest from the structure described above: page order, thumbnails, deep zoom and the table of contents.

In [ ]:
def mirador(manifest_url, canvas_id=None, height=620, view=None):
    # Build the HTML page for a Mirador 3 window that opens the given manifest.
    window = {"manifestId": manifest_url}
    if canvas_id:
        window["canvasId"] = canvas_id
    if view:
        window["view"] = view
    cfg = {"id": "viewer", "windows": [window], "window": {"allowClose": False, "allowMaximize": False},
           "workspaceControlPanel": {"enabled": False}}
    cfg_json = json.dumps(cfg).replace("</", "<\\/")
    doc = ("<!doctype html><html><head><meta charset='utf-8'>"
           "<link rel='stylesheet' href='https://fonts.googleapis.com/css?family=Roboto:300,400,500'>"
           "<style>html,body,#viewer{margin:0;height:100%;position:relative}</style></head><body>"
           "<div id='viewer'></div>"
           "<script src='https://cdn.jsdelivr.net/npm/mirador@3.4.3/dist/mirador.min.js'></script>"
           f"<script>Mirador.viewer({cfg_json});</script></body></html>")
    return doc

def show_mirador(*args, height=620, **kw):
    doc = mirador(*args, height=height, **kw)
    display(HTML(f"<iframe srcdoc=\"{html.escape(doc, quote=True)}\" "
                 f"style='width:100%;height:{height}px;border:1px solid #ccc'></iframe>"))

show_mirador(BOOK_URL)

The same manifest also opens in other viewers, such as [Universal Viewer](https://universalviewer.io/) or the viewers built into many library websites. The manifest is the shared part; each viewer interprets it in its own way.

Mirador can also display annotation pages, provided they are reachable by URL (or through its annotation plugin). That is the same step as in section 4: the viewer reads the `target`, finds the region on the canvas and draws it.

## 6. Case study: a Book of Hours assembled from separate owners

[Fragmentarium](https://fragmentarium.ms/) is a platform for medieval manuscript fragments. A common situation with such material is that one codex was taken apart, and its leaves are now held in many places. The Fragmentarium record [F-mp8l](https://fragmentarium.ms/overview/F-mp8l) is a *virtual reconstruction*: a Book of Hours (France, early 15th century) that was dismembered in the 1940s. Leaves were distributed as no. 45 in Otto Ege's portfolio *Fifty Original Leaves from Medieval Manuscripts*, and 31 of them are known today. The reconstruction was prepared in 2022 by students of Simmons University under the direction of Lisa Fagin Davis.

The reconstruction is published as a single IIIF manifest. The page linked in the workshop description, [`F-mp8l/7653/61324`](https://fragmentarium.ms/view/page/F-mp8l/7653/61324), shows one leaf of it, from the State Library of New York.

In [ ]:
FRAG_URL = "https://fragmentarium.ms/metadata/iiif/F-mp8l/manifest.json"
frag = get_json(FRAG_URL)

meta = {m["label"]: m["value"] for m in frag["metadata"]}
for key in ["Title", "Subtitle", "Document Type", "Number of Fragment Items", "Date of origin", "Place of origin"]:
    print(f"{key:26} {meta[key]}")
print(f"{'Licence':26} {frag['license']}")
print(f"{'Manifest format':26} {frag['@context']}")

canvases = frag["sequences"][0]["canvases"]
print(f"\ncanvases in the manifest: {len(canvases)}  (31 leaves, recto and verso, minus one single-sided leaf)")

This manifest uses the older Presentation 2 vocabulary (`sequences`, `canvases`, `images`, `resource`) but the structure is the same as in section 2: a list of canvases, each with an image attached.

Each canvas identifier contains a Fragmentarium record number, such as `F-s4ej`. These are the records of the **individual leaves**: every leaf has its own page, metadata (holding institution, shelfmark, description) and its own manifest. The reconstruction manifest lists their canvases in one sequence:

In [ ]:
def frag_id(canvas):
    return re.search(r"/canvas/(F-\w+)/", canvas["@id"]).group(1)

ids = []
for cv in canvases:
    fid = frag_id(cv)
    if fid not in ids:
        ids.append(fid)
print(len(ids), "leaf records:", ", ".join(ids[:8]), "...")

ny = next(cv for cv in canvases if frag_id(cv) == "F-s4ej")
print("\nOne canvas from the reconstruction manifest:")
print(json.dumps({"@id": ny["@id"], "label": ny["label"], "width": ny["width"], "height": ny["height"],
                  "image service": ny["images"][0]["resource"]["service"]["@id"]}, indent=2))

The individual leaf records each carry their own description. Fetching them gives a table of where each leaf is now held:

In [ ]:
def leaf_info(fid):
    try:
        m = get_json(f"https://fragmentarium.ms/metadata/iiif/{fid}/manifest.json")
        d = {x["label"]: x["value"] for x in m["metadata"]}
        return fid, d.get("Location", ""), d.get("Collection Name", ""), d.get("Shelfmark", "")
    except Exception as e:
        return fid, "?", "?", str(e)

with ThreadPoolExecutor(6) as ex:
    leaves = list(ex.map(leaf_info, ids))

display(HTML("<table><tr><th>record</th><th>location</th><th>collection</th><th>shelfmark</th></tr>" + "".join(
    f"<tr><td><a href='https://fragmentarium.ms/overview/{f}'>{f}</a></td><td>{html.escape(l)}</td>"
    f"<td>{html.escape(c)}</td><td>{html.escape(s)}</td></tr>" for f, l, c, s in leaves) + "</table>"))

The holders are mostly universities, public libraries and museums. Two leaves belong to a private collection that Fragmentarium lists under a placeholder name (the McBride Collection), and one leaf is recorded from an auction sale with its current location unknown. A researcher who wants to see the whole book would otherwise have to visit these records one by one. The reconstruction manifest puts their canvases into the codicological order, so that a viewer can page through them as one book.

The contact sheet below follows that order (recto and verso of each leaf side by side, reading left to right). It makes one small image request per canvas:

In [ ]:
def contact_sheet(canvases, thumb_h=170, per_row=10):
    def one(cv):
        svc = cv["images"][0]["resource"]["service"]["@id"]
        try:
            im, _ = pil(f"{svc}/full/,{thumb_h}/0/default.jpg")
            return im.convert("RGB")
        except Exception:
            return Image.new("RGB", (thumb_h * 3 // 4, thumb_h), "#ddd")
    with ThreadPoolExecutor(8) as ex:
        thumbs = list(ex.map(one, canvases))
    cw = max(t.width for t in thumbs) + 8
    rows = math.ceil(len(thumbs) / per_row)
    sheet = Image.new("RGB", (cw * per_row, (thumb_h + 8) * rows), "white")
    for i, t in enumerate(thumbs):
        sheet.paste(t, ((i % per_row) * cw + 4, (i // per_row) * (thumb_h + 8) + 4))
    return sheet

display(contact_sheet(canvases))

Two properties of the manifest are visible here. The photographs also differ in lighting, background and framing, since each owner imaged their own leaf.

**Heterogeneous images.** The leaves were photographed or scanned by different owners at different times, so the originals differ widely in resolution. Each canvas records its own dimensions and each image service reports its own size, and the viewer handles the difference:

In [ ]:
sizes = sorted(((cv["width"], cv["height"], cv["label"]) for cv in canvases), key=lambda t: t[0] * t[1])
print("smallest canvas:", sizes[0][:2], sizes[0][2])
print("largest canvas: ", sizes[-1][:2], sizes[-1][2])

**Structure on top of the sequence.** The manifest also contains *ranges*, which divide the book into its liturgical texts, as in the "Range" boxes of the diagram in section 1:

In [ ]:
by_id = {s["@id"]: s for s in frag["structures"]}
top_level = next(s for s in frag["structures"] if s.get("@viewingHint") == "top")

for rid in top_level["ranges"]:
    section = by_id[rid]
    children = [s for s in frag["structures"] if s.get("within") == rid]
    n = sum(len(c.get("canvases", [])) for c in children)
    print(f"{section['label']}  ({n} canvases)")
    for c in children:
        print(f"    - {c['label']}: {len(c.get('canvases', []))} canvases")

### What the reconstruction manifest contains, and what it does not

The manifest for F-mp8l is a small JSON file of about 60 KB. It contains:

- the metadata and the scholarly description of the reconstruction,
- the list of 61 canvases, each pointing to a leaf record and an image service,
- the ranges that group the canvases into texts.

It does not contain image data. The images are fetched from image services when a viewer asks for them. The leaf records stay separate, with their own identifiers and descriptions, so the same leaf can be referred to individually and can appear in more than one assembly.

One thing worth being precise about: in this particular case, the image services are all run by Fragmentarium itself (the identifiers point to `fragmentarium.ms/loris/…`), which holds the images that contributors uploaded. So the reconstruction avoids copying images *between manifests*, but not hosting them centrally. The mechanism does not depend on this. A canvas can point to an image service anywhere, and the diagram in section 1 shows a book whose pages come from three different institutions' own servers.

The reconstruction can be opened in Mirador, starting at the New York State Library leaf. The facing pages in the book view come from different owners:

In [ ]:
show_mirador(FRAG_URL, canvas_id=ny["@id"], view="book", height=680)

## Summary

| What the standard provides | How |
|---|---|
| One address per object | The manifest URL identifies a whole book, with its metadata and rights |
| Ordered pages, independent of file names | A list of canvases; each canvas is a coordinate space |
| Images hosted anywhere | A painting annotation connects an image service to a canvas |
| Sections and tables of contents | Ranges group canvases without reordering them |
| Any size or region of an image on request | The Image API URL: `{service}/{region}/{size}/{rotation}/{quality}.{format}` |
| Deep zoom with small transfers | Tiles are ordinary region requests at fixed scale factors |
| Notes on part of a page | Annotations target `canvas#xywh=…` and leave the image untouched |
| Assembling one object from many sources | A manifest can list canvases whose images come from different services |

### Limits to keep in mind

- Not every institution offers an image service. Some only publish static image files, in which case region and size requests are not available. Check the `service` entry before relying on them.
- Servers differ in compliance level, in API version (2 or 3) and in the formats they support. Always read `info.json`.
- A manifest is a description made by someone. Its page order, labels and groupings reflect that person's decisions, which is also true of the Fragmentarium reconstruction.

### References

- IIIF overview and specifications: <https://iiif.io/>
- Presentation API 3.0: <https://iiif.io/api/presentation/3.0/>
- Image API 3.0: <https://iiif.io/api/image/3.0/>
- IIIF Cookbook (worked manifest examples, including annotations): <https://iiif.io/api/cookbook/>
- Mirador: <https://projectmirador.org/>
- Fragmentarium: <https://fragmentarium.ms/>